# AES: ECB Leakage, CBC, and GCM Authenticated Encryption

These examples use the running scenario of St. Isidore Hospital. They are teaching examples: understand the mechanism, then prefer well-reviewed libraries and current protocols in production.

## Goal

AES is a modern symmetric block cipher. The dangerous part for students to see is that the mode matters: AES-ECB leaks repeated patterns, while authenticated modes such as AES-GCM provide confidentiality and integrity.

In [ ]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from Crypto.Util.Padding import pad, unpad

block = b"ICU-ROOM-07-ABCD"  # exactly 16 bytes
plaintext = block * 6  # Repetition makes ECB leakage visible.
key = get_random_bytes(16)  # 16 bytes = 128-bit AES key.

ecb = AES.new(key, AES.MODE_ECB)  # ECB encrypts each block independently.
ct_ecb = ecb.encrypt(plaintext)  # No padding needed: plaintext is already block-aligned.
chunks = [ct_ecb[i:i+16].hex() for i in range(0, len(ct_ecb), 16)]
print("ECB blocks:")
for c in chunks:
    print(c)
print("Unique blocks:", len(set(chunks)), "of", len(chunks))

In [ ]:
clinical_note = b"Patient 2048: discharge summary, diagnosis, medications, allergies."

cbc_key = get_random_bytes(16)  # Use a fresh key for the CBC example.
iv = get_random_bytes(16)  # CBC requires a fresh IV.
cbc = AES.new(cbc_key, AES.MODE_CBC, iv)
ct_cbc = cbc.encrypt(pad(clinical_note, AES.block_size))  # Pad because notes are not block-aligned.
recovered = unpad(AES.new(cbc_key, AES.MODE_CBC, iv).decrypt(ct_cbc), AES.block_size)
print("CBC ciphertext:", ct_cbc.hex())
print(recovered)

## AES-GCM

GCM is usually a better teaching default because it returns both ciphertext and an authentication tag. If the ciphertext or associated metadata changes, verification fails.

In [ ]:
gcm_key = get_random_bytes(32)  # 32 bytes = 256-bit AES key.
nonce = get_random_bytes(12)  # GCM commonly uses a 12-byte nonce; do not reuse it with this key.
aad = b"EHR-message:lab-result:v1"  # Associated data is authenticated but not encrypted.

gcm = AES.new(gcm_key, AES.MODE_GCM, nonce=nonce)
gcm.update(aad)  # Bind metadata to the ciphertext.
ct, tag = gcm.encrypt_and_digest(clinical_note)  # Encrypt and compute the authentication tag.
print("Ciphertext:", ct.hex())
print("Tag:", tag.hex())

verify = AES.new(gcm_key, AES.MODE_GCM, nonce=nonce)
verify.update(aad)  # Verification must use the same associated data.
print(verify.decrypt_and_verify(ct, tag))  # Raises ValueError if ciphertext or tag is invalid.

tampered = bytearray(ct)
tampered[0] ^= 1  # Flip one bit to simulate an attacker modifying the ciphertext.
try:
    verify = AES.new(gcm_key, AES.MODE_GCM, nonce=nonce)
    verify.update(aad)
    verify.decrypt_and_verify(bytes(tampered), tag)
except ValueError as exc:
    print("Tampering detected:", exc)